# Hunting MicroMonsters Bioinformatic Analysis Workflow

## Convert long *.fast5 names to short names

This document shows a method to easely convert long file names from ONT sequencing raw data to a short-consecutive-number set of files.
Each file name will be compound by the container folder name + the number of the file starting from 0.

#### Step-by-step breakdown:

1. find . -type f: Finds all files (not directories) starting from the current directory (.) and going into all subdirectories recursively.

2. while read -r filepath: Processes each file path found by the find command one at a time.

3. dir=$(dirname " $filepath"): Extracts the directory path from the full file path. For example:
    - If filepath is ./subfolder/myfile.txt, dir becomes ./subfolder
    - If filepath is ./myfile.txt, dir becomes .

4. file=$(basename "$filepath"): Extracts just the filename from the full path.
    - From ./subfolder/myfile.txt, file becomes myfile.txt
5. Folder name logic:
    - If the file is in the root directory (dir = "."), use the current working directory's name
    - Otherwise, use the subdirectory's name as the prefix
6. Counters calculation
    - Counts how many files already exist in that directory with the naming pattern foldername_*
    - This ensures proper numbering even if the script is run multiple times

7. Extension handling: Same as your original code - extracts the file extension.

8. File renaming: Moves the file to its new name with the pattern foldername_counter.extension


### Changing the name of each file in one folder with the format "folder_name + number + extension"

Change the name while keeping the original file extensions. This process uses bash kernel.

In [ ]:
#locate the folder with the files
cd ~/path/to/folder
counter=0
folder_name=$(basename "$PWD")
for file in *; do
    if [ -f "$file" ]; then
        # Get file extension
        ext="${file##*.}"
        # Only add extension if file had one
        if [ "$ext" != "$file" ]; then
            mv "$file" "${folder_name}_${counter}.${ext}"
        else
            mv "$file" "${folder_name}_${counter}"
        fi
        ((counter++))
    fi
done

### Changing the name of each file in all subdirectories of a folder with the format "folder_name + number + extension"

A modification from the previous script that can change the name of several subdirectories contained in a main folder.

In [ ]:
# locate the folder with the files
cd ~/path/to/folder

find . -type f | while read -r filepath; do
    # Get the directory and filename
    dir=$(dirname "$filepath")
    file=$(basename "$filepath")
    
    # Get folder name (use parent folder name or current folder for root)
    if [ "$dir" = "." ]; then
        folder_name=$(basename "$PWD")
    else
        folder_name=$(basename "$dir")
    fi
    
    # Count existing renamed files in this directory to continue numbering
    counter=$(find "$dir" -maxdepth 1 -name "${folder_name}_*" | wc -l)
    
    # Get file extension
    ext="${file##*.}"
    
    # Rename the file
    if [ "$ext" != "$file" ]; then
        mv "$filepath" "${dir}/${folder_name}_${counter}.${ext}"
    else
        mv "$filepath" "${dir}/${folder_name}_${counter}"
    fi
done

## Converting the fast5 files to pod5

In this section it is described the process to convert the fast5 files from the sequencing to pod5 to optimized computing performance. The data will be processed downstream using Guppy.

The general workflow for this section is:
1. Environment creation
2. Convert single fasta5 reads into multi-read fast5 reads; using the "ont-fast5-api" module tool.
3. Convert the multiread-fast5 to pod5
4. Basecalling using guppy and the pod5 files

### Fast5-to-pod5 environment creation

Instructions to create the environments and modules needed for each task of the workflow (all commands run on bash kernel).

In [ ]:
# Create the environment
conda create -n fast5-to-pod5 -y

In [ ]:
# Activate environment
conda activate fast5-to-pod5

In [ ]:
# Install ont-fast5-api tool
pip install ont-fast5-api

In [ ]:
# Install pod5 tool
pip install pod5

### Activate the fast5-to-pod5 environment which contains the pod5 module

In [ ]:
# Activate the fast5-to-pod5 conda environment if is not active yet
conda activate fast5-to-pod5

Verify that the environment is active and that the module pod5 is running by looking for the "ont-fast5-api" module

In [ ]:
conda list ont-fast5-api

### Convert single fasta5 reads into multi-read fast5

To create the multi-read files, the tool will create a single file with all the fast5 files in the working folder (4000). The tool requires the input folder path and the output folder. "--recursive" command will look in subdirectories to convert.

In [ ]:
single_to_multi_fast5 \
--input_path ~/path/to/folder/input \
--save_path ~/path/to/folder/ouptut \
--recursive

### Converting multiread-fast5 files into pod5

The newly converted multiread-fast5 will be converted to pod5 files with the pod5 tool. The script is run on bash and includes the command "--one-to-one" to keep the order of the files. The command "--threads" specify the use of all the cores available to speed up the process.

In [ ]:
pod5 convert fast5 \
~/path/to/folder/input/*.fast5 \
--output ~/path/to/folder/output \
--one-to-one ~/path/to/folder/input \
--threads 8

In [ ]:
# Deactivate the fast5-to-pod5 environment once the process is finished
conda deactivate

## Basecalling with Guppy from pod5 files

Now that we have the pod5 files on a folder, we will use Guppy tool for the basecalling. The script is run on bash using the "guppy" environment

Download the guppy file "ont_guppy_cpu_6.5.7-1~focal_amd64.deb" from the ONT community official webpage:
https://nanoporetech.com/software/other/guppy 

In [ ]:
# Create the guppy environment
conda create -n guppy

In [ ]:
# Activate the guppy environment
conda activate guppy

In [ ]:
# Install the *.deb file with guppy basecaller
sudo apt install ./ont_guppy_cpu_6.5.7-1~focal_amd64.deb

In [ ]:
# Verify that the environment is install and working by showing the "version"
guppy_basecaller --version

In [ ]:
# Check the help menu for details
guppy_basecaller -h

In [ ]:
# Execute the guppy module

guppy_basecaller \
  -i ~/path/to/folder/input/pod5_files \
  -s ~/path/to/folder/guppy_output \
  -c dna_r9.4.1_450bps_hac.cfg \
  --num_callers 4 \
  --cpu_threads_per_caller 2

Explanation of the script:

1. guppy_basecaller = calls the guppy module
2. -i = path to input folder
3. -s = path to save folder with fastq data created
4. -c = configuration of the cell flow. In this case, is an old chemistry. The "hac" indicates "high accuracy" (good quality, slow process)
5. --num_callers = number of parallel processes
6. --cpu_threads_per_caller number of cores per caller

The process using only CPU will take some days to finish, depending on the file size.

In [ ]:
# Deactivate the guppy environment
conda deactivate

## Quality analysis

This section is to verify the quality of the fastq files generated, data analysis of the summary_sequencing.txt file generated by guppy.

In [ ]:
# Install nanoplot tool
pip install NanoPlot

In [ ]:
# Create plots from sequencing using the summary file from guppy basecaller
NanoPlot \
--summary sequencing_summary.txt \
--loglength \
-o ~/path/to/folder/output 